# Potential Energy Exploration with ORCA

## Introduction
We will carry out quantum chemical calculations to study reaction kinetics and thermodynamics.
- QC and ORCA introduction given by Antti.
- We will be combining multiple computational levels to make fast calculations for larger set of conformers and accurate calculations for the best structures.

## Learning outcomes
After completing this notebook, you will be able to:
- Use various PES exporing approaches implemented in ORCA.
- Compare energies of reactants, transition states and products.
- Calculate reaction rate constants and equilibrium constants.

## Setting up the calculation environment for ORCA
Run the next code block to setup the calculation environment for ORCA.

In [ ]:
import sys
sys.path.append('../tools')
from qctools import load_xyz, load_xyz_as_traj, show_molecule

# TASK 1: Calculate equilibrium constant at 298 K for the formaldehyde–water complex formation reaction
To calculate $K$ for the reaction FA + W <=> (FA-W), we need to calculate Gibbs free formation energy for formaldehyde–water complex as well as FA and W monomers.

## Build molecular structure of formaldehyde and copy the xyz coordinates

Let's start by building formaldehyde molecular structure using Avogadro and then optimize it at DFT. The water DFT structure has been created in the QC-basic exercise.

In [ ]:
%%writefile fa.xyz
4
fa
  C           0.000      0.000      0.000
  O           1.193      0.000     -0.000
  H          -0.597      0.927     -0.000
  H          -0.597     -0.927     -0.000

In [ ]:
%%writefile dft_fa.inp
! PBE def2-TZVP opt freq

* xyzfile 0 1 fa.xyz

In [ ]:
!$ORCA_HOME/orca dft_fa.inp > dft_fa.out
!grep "TERMINATED" dft_fa.out
!grep "TOTAL RUN TIME" dft_fa.out

## Build a complex structure: formaldehyde + water
To calculate the interaction energy between formaldehyde and water molecule, we want to find the global minimum free energy structure of fa-w complex.
We are using Automated docking (DOCKER) tool as implemented in ORCA.

In DOCKER approach, monomer structures of ga and w should be given, and DOCKER is building ga-w complex structures by putting these monomers together -> hopefully also yielding to global minimum structure! Let's use very fast XTB method to go through the potential energy surface.

Write a docker input structure and run it.

In [ ]:
%%writefile docker_fa-w.inp
! XTB PAL4

%DOCKER
     GUEST "h2o.xyz"
END

*XYZFILE 0 1 dft_fa.xyz

In [ ]:
!$ORCA_HOME/orca docker_fa-w.inp > docker_fa-w.out
!grep "TERMINATED" docker_fa-w.out
!grep "TOTAL RUN TIME" docker_fa-w.out

In [ ]:
docker = load_xyz_as_traj('docker_fa-w.docker.struc1.all.optimized.xyz')
show_molecule(docker)

## Now there are several minimum energy structures and we should find the lowest one. XTB provides reasonable structures, but more accurate method should be used further. Let's reoptimize all of the XTB structures by using fast composite method B97-3c.

In [ ]:
!cp docker_fa-w.docker.struc1.all.optimized.xyz complex.allxyz

In [ ]:
%%writefile SLOPPYopt_complex.inp
! B97-3c LooseOPT SloppySCF PAL4
* xyzfile 0 1 complex.allxyz

In [ ]:
!$ORCA_HOME/orca SLOPPYopt_complex.inp > SLOPPYopt_complex.out
!grep "TERMINATED" SLOPPYopt_complex.out
!grep "TOTAL RUN TIME" SLOPPYopt_complex.out

## Select the global minimum energy structure and use DFT to reoptimize geometry and calculate frequencies.
You can use command: !cp NameOfYourMinimumEnergyFile.xyz dft_complex.xyz

In [ ]:
%%writefile dft_complex.inp
! PBE def2-TZVP opt freq

* xyzfile 0 1 dft_complex.xyz

In [ ]:
!$ORCA_HOME/orca dft_complex.inp > dft_complex.out
!grep "TERMINATED" dft_complex.out
!grep "TOTAL RUN TIME" dft_complex.out

### If you have lost the water DFT output file, then perform the following calculation (NOTE: it is important that all final calculations are done at the same level of theory!)

In [ ]:
%%writefile dft_h2o.inp
! PBE def2-TZVP opt freq

* xyzfile 0 1 h2o.xyz

In [ ]:
!$ORCA_HOME/orca dft_h2o.inp > dft_h2o.out
!grep "TERMINATED" dft_h2o.out
!grep "TOTAL RUN TIME" dft_h2o.out

In [ ]:
!grep "Final Gibbs free energy" *.out

## -> Calculate $\Delta G$ for the compex formation and use it to calculate $K$.






 

## TASK 2: Calculate reaction energy of methanediol formation.
2a) Calculate equilibrium constant for the reaction of (FA-W) <=> methanediol at 298 K

In [ ]:
%%writefile methanediol.inp
! XTB goat
* xyz 0 1 
C      0.0000   0.0000   0.0000
O      1.3000   0.0000   0.0000
O     -1.3000   0.0000   0.0000
H      0.0000   1.0000   0.0000
H      0.0000  -1.0000   0.0000
H      1.8000   0.8000   0.0000
H     -1.8000   0.8000   0.0000
*

In [ ]:
!$ORCA_HOME/orca methanediol.inp > methanediol.out
!grep "TERMINATED" methanediol.out
!grep "TOTAL RUN TIME" methanediol.out

In [ ]:
goat = load_xyz_as_traj('methanediol.finalensemble.xyz')
show_molecule(goat)

# Again there are several minimum energy structures and we should find the lowest one. Do reoptimization using fast composite method.

In [ ]:
!cp methanediol.finalensemble.xyz methanediol.allxyz

In [ ]:
%%writefile opt_methanediol.inp
! B97-3c LooseOPT SloppySCF PAL4
* xyzfile 0 1 methanediol.allxyz

In [ ]:
!$ORCA_HOME/orca opt_methanediol.inp > opt_methanediol.out
!grep "TERMINATED" opt_methanediol.out
!grep "TOTAL RUN TIME" opt_methanediol.out

# Select the lowest energy structure for the higher level of theory calculation. Optimize structure and calculate frequencies at DFT.
You can use the command !cp YourGlobalMinimum.xyz dft_methanediol.xyz

In [ ]:
%%writefile dft_methanediol.inp
! PBE def2-TZVP OPT FREQ

* xyzfile 0 1 dft_methanediol.xyz

In [ ]:
!$ORCA_HOME/orca dft_methanediol.inp > dft_methanediol.out
!grep "TERMINATED" dft_methanediol.out
!grep "TOTAL RUN TIME" dft_methanediol.out

In [ ]:
!grep "Final Gibbs free energy" *.out

## -> Calculate $\Delta G$ for the reaction and use it to calculate $K$.

## TASK 3: calculate barrier energy for methanediol formation
3a) Calculate bimolecular reaction rate constant for the reaction of FA + W <=> methanediol at 298 K

First we need to locate transition state structure. This can be done using Nudged Elastic Band with TS optimization (NEB-TS) method implemented in ORCA.

We need to have reactant complex and product structures. NOTE that the atom numbering must me same in reactant and complex – you might have to manually change that.

In [ ]:
%%writefile b97_neb.inp
! B97-3c NEB-TS FREQ PAL4
    
%NEB 
 NEB_END_XYZFILE "b97_methanediol.xyz" 
END

*XYZFILE 0 1 b97_complex.xyz

In [ ]:
!$ORCA_HOME/orca b97_neb.inp > b97_neb.out
!grep "TERMINATED" b97_neb.out
!grep "TOTAL RUN TIME" b97_neb.out

In [ ]:
!cp b97_neb_NEB-TS_converged.xyz dft_TS.xyz

In [ ]:
%%writefile dft_TS.inp
! PBE def2-TZVP OptTS FREQ

* xyzfile 0 1 dft_TS.xyz

In [ ]:
!$ORCA_HOME/orca dft_TS.inp > dft_TS.out
!grep "TERMINATED" dft_TS.out
!grep "TOTAL RUN TIME" dft_TS.out

In [ ]:
!grep "Final Gibbs free energy" *.out

## -> Calculate $\Delta G^{barrier}$ and use it to calculate $k_{bi}$.

## Additional tasks if time allows:
- transition state sampling
- higher level energy connections
- effect of a second water molecule as a catalyst